# 关于Qwen3.5技术报告的解读
## 架构
![structure](pic/qwen3.png)
## 亮点

* MRoPE多模态（时间、高度、宽度）旋转位置编码
* Gated Attention（2025年NIPS最佳论文）
    -> NIPS最近发生的小风波大家知道吗？
* 混合注意力

## LLM
### Gated DeltaNet + Gated Attention 混合注意力

![ha](pic/hybridAttention.png)

#### O(N^2)的时间复杂度
非混合注意力的模型，我们较为熟知的，比如LLaMa等，注意力层使用的是传统的点积注意力的变体。对于传统的点积注意力，每一次进行注意力分数的计算的时间复杂度是 $ O(N^2) $ 的，即，有N个token的情况下，需要做$ N^2 $ 次运算。
这里给出torch风格的关键运算步骤的伪代码：


In [ ]:
import torch
import torch.nn as nn
import math

class self_attention(nn.Module):
    def __init__(self, input_dim) -> None:
        super().__init__()
        self.embedding_dim = input_dim
        self.attention_dim = input_dim
        self.q = nn.Linear(self.embedding_dim, self.attention_dim)
        self.k = nn.Linear(self.embedding_dim, self.attention_dim)
        self.v = nn.Linear(self.embedding_dim, self.attention_dim)
    def forward(self, x):# x.shape = bs, seqlen, input_dim 
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        # 这里q矩阵与k矩阵进行矩阵乘法，q和k的大小为均为 $ N \times d $
        # 那么scores的大小就会是 $ N \times N $
        # 因此N方的时间复杂度便是从这里体现的
        scores = torch.bmm(q, k.transpose(1, 2)) / math.sqrt(self.attention_dim)
        weights = torch.softmax(scores, dim=-1)


        out = torch.bmm(weights, v)
        return out

In [ ]:
bs, seqlen, input_dim = 2, 10, 64
x = torch.rand(bs, seqlen, input_dim)
attn = self_attention(input_dim)
output = attn(x)
print("x.shape:", x.shape)
print("out.shape:", output.shape)


这里q矩阵与k矩阵进行矩阵乘法，q和k的大小为均为 $ N \times d $，那么scores的大小就会是 $ N \times N $， 因此N方的时间复杂度便是从这里体现的。

#### O(N)的时间复杂度，Linear！
研究者们思考，能否以O(N)的时间复杂度计算注意分数呢？这引发了18年以来的线性注意力机制相关的研究，例如Mamba等等模型。这里给出几篇优质的文献作为参考：
苏剑林的线性注意力系列

>苏剑林. (Jun. 20, 2025). 《线性注意力简史：从模仿、创新到反哺 》[Blog post]. Retrieved from https://spaces.ac.cn/archives/11033

>苏剑林. (Jul. 04, 2020). 《线性Attention的探索：Attention必须有个Softmax吗？ 》[Blog post]. Retrieved from https://spaces.ac.cn/archives/7546

Gated DeltaNet原论文 *Gated Delta Networks: Improving Mamba2 with Delta Rule*

>arXiv:2412.06464 [cs.CL]

#### 对于混合注意力架构的一点理解
* 线性注意力 -> 更快的计算速度 -> 更高的处理效率 -> 更大的参数规模
* Qwen3.5的混合注意力层为线性注意力比全注意力等于3:1，如此混合有效结合了线性注意力的计算效率优点与全注意力的全局信息捕捉能力优点。
* 本质上属于对模型结构的边界探索，即，在参数量确定的情况下，怎样的模型架构会使得模型拥有更高的智能密度上限？